# Start Partition Comparison

This notebook compares the available start partition algorithms for the Dense Graph Partition experiments.

The comparison is performed separately for:

- Powerlaw and Erdős–Rényi graphs,
- sparse and dense instances,
- small and large instances.

Only two aggregated metrics are reported:

- **mean relative to best**: mean quotient between the solution density of an algorithm and the best density found on the same instance;
- **mean runtime**: mean runtime in seconds.

A value close to `1.0` for the relative solution quality indicates that an algorithm produces solutions close to the best available result.

In [27]:
from pathlib import Path

import numpy as np
import pandas as pd

In [28]:
RESULTS_FILE = Path("../results/experiment1/raw_results.csv")

ALGORITHM_ORDER = [
    "singleton",
    "matching",
    "maximum_matching",
    "high_degree_first_matching",
    "high_degree_product_matching",
    "kapoce",
    "leiden",
]

GRAPH_ORDER = ["powerlaw", "er"]
REGIME_ORDER = ["sparse", "dense"]
SIZE_ORDER = ["small", "large"]

## Load experiment results

The raw experiment results are loaded and checked for the columns required by this analysis.

In [29]:
raw = pd.read_csv(RESULTS_FILE)

required_columns = {
    "graph_type",
    "regime",
    "size_class",
    "algorithm",
    "relative_to_best",
    "runtime",
}

missing_columns = required_columns.difference(raw.columns)

if missing_columns:
    raise ValueError(
        "The result file is missing required columns: "
        + ", ".join(sorted(missing_columns))
    )

print(f"Loaded {len(raw):,} result rows.")
print("Algorithms:", ", ".join(sorted(raw["algorithm"].unique())))
raw.head()

Loaded 14,000 result rows.
Algorithms: high_degree_first_matching, high_degree_product_matching, kapoce, leiden_mdgp, matching, maximum_matching, singleton


,dataset,size_class,graph_type,regime,instance,n,m,edge_density,algorithm,density,num_clusters,max_cluster_size,avg_cluster_size,runtime,relative_to_best,is_best
0,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_003_n52_s1218,52,100,0.0754,singleton,0.0,52,1,1.0000,0.0000,0.0000,False
1,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_003_n52_s1218,52,100,0.0754,matching,8.0,36,2,1.4444,0.0003,0.6486,False
2,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_003_n52_s1218,52,100,0.0754,maximum_matching,11.5,29,2,1.7931,0.0016,0.9324,False
3,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_003_n52_s1218,52,100,0.0754,high_degree_first_matching,8.0,36,2,1.4444,0.0002,0.6486,False
4,powerlaw_sparse_small,small,powerlaw,sparse,powerlaw_sparse_small_003_n52_s1218,52,100,0.0754,high_degree_product_matching,8.0,36,2,1.4444,0.0001,0.6486,False


## Aggregate solution quality and runtime

For every combination of graph type, density regime, size class, and start partition algorithm, the notebook calculates:

1. the mean relative solution quality;
2. the mean runtime in seconds.

Each instance contributes one observation to the corresponding group.

In [30]:
summary = (
    raw
    .groupby(
        ["graph_type", "regime", "size_class", "algorithm"],
        as_index=False,
        observed=True,
    )
    .agg(
        mean_relative_to_best=("relative_to_best", "mean"),
        mean_runtime_seconds=("runtime", "mean"),
    )
)

summary["graph_type"] = pd.Categorical(
    summary["graph_type"],
    categories=GRAPH_ORDER,
    ordered=True,
)
summary["regime"] = pd.Categorical(
    summary["regime"],
    categories=REGIME_ORDER,
    ordered=True,
)
summary["size_class"] = pd.Categorical(
    summary["size_class"],
    categories=SIZE_ORDER,
    ordered=True,
)

present_algorithms = summary["algorithm"].unique().tolist()
algorithm_order = [
    algorithm
    for algorithm in ALGORITHM_ORDER
    if algorithm in present_algorithms
]
algorithm_order += sorted(
    set(present_algorithms).difference(algorithm_order)
)

summary["algorithm"] = pd.Categorical(
    summary["algorithm"],
    categories=algorithm_order,
    ordered=True,
)

summary = (
    summary
    .sort_values(["graph_type", "size_class", "regime", "algorithm"])
    .reset_index(drop=True)
)

summary

,graph_type,regime,size_class,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,sparse,small,singleton,0.000000,0.000034
1,powerlaw,sparse,small,matching,0.716396,0.000284
2,powerlaw,sparse,small,maximum_matching,0.956452,0.015091
3,powerlaw,sparse,small,high_degree_first_matching,0.678270,0.001403
4,powerlaw,sparse,small,high_degree_product_matching,0.678270,0.000691
5,powerlaw,sparse,small,kapoce,0.995078,0.032730
6,powerlaw,sparse,small,leiden_mdgp,0.966590,0.000973
7,powerlaw,dense,small,singleton,0.000000,0.000026
8,powerlaw,dense,small,matching,0.705559,0.000310
9,powerlaw,dense,small,maximum_matching,0.898826,0.018099


In [31]:
final_table = summary[
    [
        "graph_type",
        "size_class",
        "regime",
        "algorithm",
        "mean_relative_to_best",
        "mean_runtime_seconds",
    ]
].copy()

final_table["mean_relative_to_best"] = (
    final_table["mean_relative_to_best"].round(4)
)
final_table["mean_runtime_seconds"] = (
    final_table["mean_runtime_seconds"].round(5)
)

final_table

,graph_type,size_class,regime,algorithm,mean_relative_to_best,mean_runtime_seconds
0,powerlaw,small,sparse,singleton,0.0000,0.00003
1,powerlaw,small,sparse,matching,0.7164,0.00028
2,powerlaw,small,sparse,maximum_matching,0.9565,0.01509
3,powerlaw,small,sparse,high_degree_first_matching,0.6783,0.00140
4,powerlaw,small,sparse,high_degree_product_matching,0.6783,0.00069
5,powerlaw,small,sparse,kapoce,0.9951,0.03273
6,powerlaw,small,sparse,leiden_mdgp,0.9666,0.00097
7,powerlaw,small,dense,singleton,0.0000,0.00003
8,powerlaw,small,dense,matching,0.7056,0.00031
9,powerlaw,small,dense,maximum_matching,0.8988,0.01810


## LaTeX helper functions

The following functions format algorithm names and numerical values for the thesis table. Values are truncated rather than rounded, matching the formatting used in the move-operator notebook.

In [32]:
def truncate_number(value: float, decimals: int) -> float:
    factor = 10 ** decimals
    return np.trunc(value * factor) / factor


def latex_algorithm(algorithm: str) -> str:
    return r"\texttt{" + algorithm.replace("_", r"\_") + "}"


def format_number(value: float, decimals: int) -> str:
    return f"{truncate_number(value, decimals):.{decimals}f}"

## Build LaTeX comparison table

The table is grouped by graph type and dataset configuration. Within each dataset group:

- the highest mean relative solution quality is printed in bold

Ties are highlighted for all affected algorithms.

In [33]:
def make_start_partition_latex_table(
        df: pd.DataFrame,
        graph_type: str,
        caption: str,
        label: str,
) -> str:
    dataset_order = [
        ("small", "sparse"),
        ("small", "dense"),
        ("large", "sparse"),
        ("large", "dense"),
    ]

    graph_df = df[df["graph_type"] == graph_type].copy()

    if graph_df.empty:
        raise ValueError(
            f"No results available for graph type '{graph_type}'."
        )

    lines = [
        r"\begin{table}[t]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        (
            r"\begin{tabular}{"
            r"p{1.9cm}"
            r"p{7cm}"
            r"p{2.4cm}"
            r"p{2.4cm}"
            r"}"
        ),
        r"\toprule",
        (
            r"Datensatz "
            r"& Startpartition "
            r"& \centering Mittlere relative Güte "
            r"& \centering\arraybackslash Mittlere Laufzeit (s) \\"
        ),
        r"\midrule",
    ]

    nonempty_datasets = [
        (size_class, regime)
        for size_class, regime in dataset_order
        if not graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].empty
    ]

    for dataset_index, (size_class, regime) in enumerate(
            nonempty_datasets
    ):
        part = graph_df[
            (graph_df["size_class"] == size_class)
            & (graph_df["regime"] == regime)
            ].copy()

        part["algorithm"] = pd.Categorical(
            part["algorithm"],
            categories=algorithm_order,
            ordered=True,
        )
        part = part.sort_values("algorithm")

        best_quality = part["mean_relative_to_best"].max()
        best_runtime = part["mean_runtime_seconds"].min()

        dataset_label = f"{size_class} {regime}"

        for row_index, row in enumerate(part.itertuples(index=False)):
            dataset_cell = (
                rf"\multirow{{{len(part)}}}{{*}}{{{dataset_label}}}"
                if row_index == 0
                else ""
            )

            quality = format_number(
                row.mean_relative_to_best,
                4,
            )
            runtime = format_number(
                row.mean_runtime_seconds,
                5,
            )

            if np.isclose(
                    row.mean_relative_to_best,
                    best_quality,
            ):
                quality = rf"\textbf{{{quality}}}"

            lines.append(
                f"{dataset_cell} "
                f"& {latex_algorithm(str(row.algorithm))} "
                f"& {quality} "
                f"& {runtime} \\\\"
            )

        if dataset_index < len(nonempty_datasets) - 1:
            lines.append(r"\cmidrule(l){1-4}")

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )

    return "\n".join(lines)

In [34]:
powerlaw_latex = make_start_partition_latex_table(
    final_table,
    graph_type="powerlaw",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der "
        "Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative "
        "Lösungsqualität ist der Quotient zur besten auf derselben "
        "Instanz gefundenen Lösung."
    ),
    label="tab:start_partition_powerlaw",
)

print(powerlaw_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Powerlaw-Instanzen. Die relative Lösungsqualität ist der Quotient zur besten auf derselben Instanz gefundenen Lösung.}
\label{tab:start_partition_powerlaw}
\begin{tabular}{p{1.9cm}p{7cm}p{2.4cm}p{2.4cm}}
\toprule
Datensatz & Startpartition & \centering Mittlere relative Güte & \centering\arraybackslash Mittlere Laufzeit (s) \\
\midrule
\multirow{7}{*}{small sparse} & \texttt{singleton} & 0.0000 & 0.00003 \\
 & \texttt{matching} & 0.7164 & 0.00027 \\
 & \texttt{maximum\_matching} & 0.9565 & 0.01509 \\
 & \texttt{high\_degree\_first\_matching} & 0.6783 & 0.00140 \\
 & \texttt{high\_degree\_product\_matching} & 0.6783 & 0.00069 \\
 & \texttt{kapoce} & \textbf{0.9951} & 0.03273 \\
 & \texttt{leiden\_mdgp} & 0.9666 & 0.00097 \\
\cmidrule(l){1-4}
\multirow{7}{*}{small dense} & \texttt{singleton} & 0.0000 & 0.00003 \\
 & \texttt{matching} & 0.7056 & 0.00031 \\
 & \text

In [35]:
er_latex = make_start_partition_latex_table(
    final_table,
    graph_type="er",
    caption=(
        "Mittlere relative Lösungsqualität und mittlere Laufzeit der "
        "Startpartitionsverfahren auf Erdős--Rényi-Instanzen. Die "
        "relative Lösungsqualität ist der Quotient zur besten auf "
        "derselben Instanz gefundenen Lösung."
    ),
    label="tab:start_partition_er",
)

print(er_latex)

\begin{table}[t]
\centering
\caption{Mittlere relative Lösungsqualität und mittlere Laufzeit der Startpartitionsverfahren auf Erdős--Rényi-Instanzen. Die relative Lösungsqualität ist der Quotient zur besten auf derselben Instanz gefundenen Lösung.}
\label{tab:start_partition_er}
\begin{tabular}{p{1.9cm}p{7cm}p{2.4cm}p{2.4cm}}
\toprule
Datensatz & Startpartition & \centering Mittlere relative Güte & \centering\arraybackslash Mittlere Laufzeit (s) \\
\midrule
\multirow{7}{*}{small sparse} & \texttt{singleton} & 0.0000 & 0.00003 \\
 & \texttt{matching} & 0.8075 & 0.00027 \\
 & \texttt{maximum\_matching} & 0.8981 & 0.01496 \\
 & \texttt{high\_degree\_first\_matching} & 0.7399 & 0.00132 \\
 & \texttt{high\_degree\_product\_matching} & 0.7399 & 0.00065 \\
 & \texttt{kapoce} & \textbf{1.0000} & 0.00827 \\
 & \texttt{leiden\_mdgp} & 0.9126 & 0.00082 \\
\cmidrule(l){1-4}
\multirow{7}{*}{small dense} & \texttt{singleton} & 0.0000 & 0.00054 \\
 & \texttt{matching} & 0.7559 & 0.00036 \\
 & \texttt